# Eco-Travel Advisor — Setup & Demo

**Conversational Agent for Sustainable Tourism Planning (Rasa + NeonDB)**

This notebook is the *setup, data and testing* layer for the project. It does **not** run the
chatbot itself — the Rasa assistant runs as a server and is deployed separately (see below).
What this notebook does, with reproducible outputs for the report:

1. Detect the environment (Google Colab or local Jupyter) and prepare the project.
2. Install dependencies.
3. Load secrets (NeonDB URL, Climatiq key) safely — values are never printed.
4. Validate the curated mock seed data.
5. Test the NeonDB connection.
6. Seed the database and demonstrate idempotent re-runs.
7. *(Later)* Fallback-logic test, Climatiq API test, and demo outputs.

### How to run the actual chatbot
- **Primary:** open the live **HuggingFace Spaces** URL and chat — no setup required.
- **Alternative:** clone the GitHub repository and run `docker compose up` (see README).
- **This notebook:** run the cells top-to-bottom to reproduce the data and test layers.

> The notebook is portable: it works unchanged in Colab and in local Jupyter.

## 1. Environment detection

In [ ]:
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environment:', 'Google Colab' if IN_COLAB else 'local Jupyter')

## 2. Prepare the project

In Colab the repository is cloned from GitHub. Locally the notebook already sits at the
project root, so this cell only resolves the path.

**Colab users:** set `REPO_URL` to your own repository before running.

In [ ]:
# Colab users: set this to your own repository before running.
REPO_URL = 'https://github.com/<your-username>/eco-travel-advisor.git'
PROJECT_DIR = 'eco-travel-advisor'

# If we are already inside the project folder (local Jupyter, or a re-run),
# do not clone again — just use the current directory.
if os.path.isdir('actions') and os.path.isdir('data/seed'):
    PROJECT_ROOT = os.getcwd()
elif IN_COLAB:
    if '<your-username>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell.')
    if not os.path.isdir(PROJECT_DIR):
        !git clone $REPO_URL
    if not os.path.isdir(PROJECT_DIR):
        raise FileNotFoundError('git clone did not create the project folder. '
                                'Check REPO_URL and that the repository is public.')
    os.chdir(PROJECT_DIR)
    PROJECT_ROOT = os.getcwd()
else:
    raise FileNotFoundError('Open this notebook from the project root '
                            '(the folder containing actions/ and data/seed/).')

print('Project root:', PROJECT_ROOT)

## 3. Install dependencies

In [ ]:
%pip install -q sqlalchemy "psycopg[binary]" python-dotenv requests

## 4. Load secrets safely

- **Colab:** add `NEON_DATABASE_URL` (and optionally `CLIMATIQ_API_KEY`) in the left-hand
  **Secrets** panel and enable notebook access.
- **Local:** put them in a `.env` file at the project root (never committed).

Only a boolean *configured?* flag is printed — the actual values are never shown.

In [ ]:
def load_secrets():
    if IN_COLAB:
        from google.colab import userdata
        for key in ('NEON_DATABASE_URL', 'CLIMATIQ_API_KEY'):
            try:
                value = userdata.get(key)
                if value:
                    os.environ[key] = value
            except Exception:
                pass  # secret not set / access not granted
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            print('python-dotenv not installed; relying on shell environment.')

load_secrets()
print('NEON_DATABASE_URL configured:', bool(os.environ.get('NEON_DATABASE_URL')))
print('CLIMATIQ_API_KEY  configured:', bool(os.environ.get('CLIMATIQ_API_KEY')))

## 5. Validate the curated mock seed data

Checks that every seed file is valid JSON and internally consistent: foreign keys resolve,
sustainability tags exist, every transport mode has an emission factor, and the emissions
figures match `distance x factor`.

In [ ]:
import json, glob

seed = {os.path.basename(p): json.load(open(p, encoding='utf-8'))
        for p in glob.glob('data/seed/*.json')}
print('Seed files loaded:', len(seed))

dest_ids = {d['destination_id'] for d in seed['destination.json']}
tag_names = {t['tag_name'] for t in seed['tags.json']}
factor_modes = {e['mode'] for e in seed['emission_factor.json']}
factor = {e['mode']: e['kg_co2e_per_passenger_km'] for e in seed['emission_factor.json']}
errors = []

for fn in ('hotel.json', 'experience.json', 'transport_option.json', 'offset_option.json'):
    for row in seed[fn]:
        if row['destination_id'] not in dest_ids:
            errors.append(f"{fn}: bad destination_id {row['destination_id']}")
        for tg in row.get('sustainability_tags', []):
            if tg not in tag_names:
                errors.append(f"{fn}: unknown tag '{tg}'")

for row in seed['transport_option.json']:
    if row['mode'] not in factor_modes:
        errors.append(f"transport_option.json: mode '{row['mode']}' has no emission factor")
    expected = round(row['estimated_distance_km'] * factor[row['mode']], 1)
    if abs(expected - row['estimated_emissions_kg_per_person']) > 0.2:
        errors.append(f"option {row['option_id']}: emissions mismatch")

for d in sorted(dest_ids):
    h = sum(1 for x in seed['hotel.json'] if x['destination_id'] == d)
    e = sum(1 for x in seed['experience.json'] if x['destination_id'] == d)
    print(f'  destination {d}: {h} hotels, {e} experiences')

print('\nVALIDATION:', 'PASSED' if not errors else f'{len(errors)} ERROR(S): {errors}')

## 6. Test the NeonDB connection

Imports the ORM models from `actions/db.py` and runs a trivial `SELECT 1`. Connection
details are never displayed. If the database is unreachable, the project still works on the
local JSON fallback (demonstrated later).

In [ ]:
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'actions'))
import db

print('DB configured:', db.is_db_configured())
if db.is_db_configured():
    from sqlalchemy import text
    try:
        with db.get_engine().connect() as conn:
            conn.execute(text('SELECT 1'))
        print('NeonDB connection: OK')
    except Exception as exc:
        print('NeonDB connection FAILED:', type(exc).__name__)
else:
    print('NEON_DATABASE_URL not set — skipping (JSON fallback will be used).')

## 7. Seed the database

Runs `actions/seed_db.py`, which creates the tables and idempotently upserts every seed
file. On the **first** run every table reports `inserted`.

In [ ]:
import subprocess

def run_seed():
    result = subprocess.run([sys.executable, 'actions/seed_db.py'],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('--- stderr ---')
        print(result.stderr)
    return result.returncode

run_seed()

## 8. Demonstrate idempotency

Re-running the seeder must **not** create duplicates: every table should now report
`updated` with `inserted = 0`. This screenshot is useful evidence in the testing section.

In [ ]:
run_seed()

## 9. Fallback-logic test  *(enabled after `actions/repository.py`)*

Will demonstrate the cascade **external API -> NeonDB -> local JSON** by forcing each tier
off in turn and confirming the same query still returns valid results, with the `data_source`
flag changing accordingly.

## 10. Climatiq API test  *(enabled after `actions/carbon.py`)*

Will call Climatiq for a sample route, show the live carbon estimate, then disable the key
to show the seamless fallback to the stored emission factors.

## 11. Demo outputs for the report  *(to be added)*

Worked end-to-end examples (e.g. *London -> Copenhagen, 2 travellers, lowest-carbon
preference*) with the carbon estimate, colour-coded recommendations and offset suggestion,
captured as figures for the final report.